# Demo: BERTweet Sentiment Inference (Seed 3 Comparison)
This notebook loads multiple sentiment checkpoints from Hugging Face, evaluates test accuracy, F1, recall, and precision on the Sentiment140 test split, and predicts sentiment for custom user text.

Models included in this demo:
- `BrianLeung/bertweet-sentiment140-full-finetune` (`seed3`)
- `BrianLeung/bertweet-sentiment140-lora` (`r32/seed3`, `r8/seed3`, `r4/seed3`)
- `BrianLeung/bertweet-sentiment140-prompt-tuning` (`seed3`)
- `BrianLeung/bertweet-sentiment140-classifier-only` (`seed3`)

The dataset loading and split logic now match the Full_Tuning notebook exactly:
- Load CSV from Google Drive path: `/content/drive/MyDrive/Copy of Sentiment140.csv`
- Filter labels to only `0` and `4`
- Relabel to binary: `0 -> 0 (negative)`, `4 -> 1 (positive)`
- Normalize mentions and URLs
- Split with seed `3` using 90/10, then split the 10% equally into validation/test

In [1]:
%pip uninstall -y torchao torchvision torchaudio
%pip install -U transformers datasets evaluate peft

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Fou

In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftConfig, PeftModel
from datasets import load_dataset
import evaluate
import torch
import re
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_SPECS = [
    {
        "display_name": "full_finetune_seed3",
        "repo": "BrianLeung/bertweet-sentiment140-full-finetune",
        "subfolder": "seed3",
        "model_type": "full",
    },
    {
        "display_name": "lora_r32_seed3",
        "repo": "BrianLeung/bertweet-sentiment140-lora",
        "subfolder": "r32/seed3",
        "model_type": "peft",
    },
    {
        "display_name": "lora_r8_seed3",
        "repo": "BrianLeung/bertweet-sentiment140-lora",
        "subfolder": "r8/seed3",
        "model_type": "peft",
    },
    {
        "display_name": "lora_r4_seed3",
        "repo": "BrianLeung/bertweet-sentiment140-lora",
        "subfolder": "r4/seed3",
        "model_type": "peft",
    },
    {
        "display_name": "prompt_tuning_seed3",
        "repo": "BrianLeung/bertweet-sentiment140-prompt-tuning",
        "subfolder": "seed3",
        "model_type": "peft",
    },
    {
        "display_name": "classifier_only_seed3",
        "repo": "BrianLeung/bertweet-sentiment140-classifier-only",
        "subfolder": "seed3",
        "model_type": "full",
    },
]

def load_tokenizer_for_spec(spec):
    return AutoTokenizer.from_pretrained("vinai/bertweet-base")

def load_model_and_tokenizer(spec):
    tokenizer = load_tokenizer_for_spec(spec)

    if spec["model_type"] == "full":
        model = AutoModelForSequenceClassification.from_pretrained(
            spec["repo"],
            subfolder=spec["subfolder"],
            attn_implementation="eager",
        )
    elif spec["model_type"] == "peft":
        peft_config = PeftConfig.from_pretrained(
            spec["repo"],
            subfolder=spec["subfolder"],
        )
        base_model = AutoModelForSequenceClassification.from_pretrained(
            peft_config.base_model_name_or_path,
            num_labels=2,
            attn_implementation="eager",
        )
        model = PeftModel.from_pretrained(
            base_model,
            spec["repo"],
            subfolder=spec["subfolder"],
        )
    else:
        raise ValueError(f"Unknown model_type: {spec['model_type']}")

    if hasattr(model, "get_input_embeddings"):
        embedding_layer = model.get_input_embeddings()
        if embedding_layer is not None:
            embedding_size = embedding_layer.num_embeddings
            tokenizer_size = len(tokenizer)
            if tokenizer_size > embedding_size:
                model.resize_token_embeddings(tokenizer_size)

    if getattr(model.config, "pad_token_id", None) is None and getattr(tokenizer, "pad_token_id", None) is not None:
        model.config.pad_token_id = tokenizer.pad_token_id

    if hasattr(model.config, "_attn_implementation"):
        model.config._attn_implementation = "eager"

    model = model.to(device)
    model.eval()
    return model, tokenizer

In [5]:
temp = load_dataset("BrianLeung/sentiment140-csv", encoding="ISO-8859-1")["train"]

temp = temp.remove_columns(["ID", "Date", "Flag", "User"])
temp = temp.filter(lambda x: x["Label"] in [0, 4])

def relabel(batch):
    labels = batch["Label"]
    new_labels = [0 if l == 0 else 1 if l == 4 else None for l in labels]
    if any(l is None for l in new_labels):
        raise ValueError("Unexpected label in batch")
    return {"Label": new_labels}

def replace(text):
    text = re.sub(r"@\w+", "@USER", text)
    text = re.sub(r"http\S+|www\.\S+", "HTTPURL", text)
    return text

def clean_text(batch):
    return {"Text": [replace(t) for t in batch["Text"]]}

temp = temp.map(relabel, batched=True)
temp = temp.map(clean_text, batched=True)
temp = temp.train_test_split(test_size=0.1, seed=3)
test_valid = temp["test"].train_test_split(test_size=0.5, seed=3)

dataset_dict = {
    "train": temp["train"].shuffle(seed=3).select(range(1)),
    "test": test_valid["test"].select(range(200)),
    "validation": test_valid["train"].select(range(1)),
}

for split_name, split_data in dataset_dict.items():
    print(split_name, len(split_data))

Map:   0%|          | 0/1600000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1600000 [00:00<?, ? examples/s]

train 1
test 200
validation 1


In [6]:
id2label = {0: "negative", 1: "positive"}

def get_prompt_virtual_tokens(model):
    peft_cfg = getattr(model, "peft_config", None)
    if not isinstance(peft_cfg, dict):
        return 0

    max_virtual_tokens = 0
    for cfg in peft_cfg.values():
        nvt = int(getattr(cfg, "num_virtual_tokens", 0) or 0)
        if nvt > max_virtual_tokens:
            max_virtual_tokens = nvt
    return max_virtual_tokens

def get_model_length_info(model):
    max_positions = getattr(model.config, "max_position_embeddings", None)
    if max_positions is None:
        return 128, None, 0

    num_virtual_tokens = get_prompt_virtual_tokens(model)
    max_input_len = max_positions - 2 - num_virtual_tokens
    if max_input_len <= 0:
        raise ValueError(
            f"max_input_len={max_input_len} is invalid (max_positions={max_positions}, num_virtual_tokens={num_virtual_tokens})."
        )
    return max_input_len, max_positions, num_virtual_tokens

def get_max_input_len(model):
    max_input_len, _, _ = get_model_length_info(model)
    return max_input_len

def predict_batch(model, tokenizer, texts, max_length):
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

    if "input_ids" in encoded:
        max_token_id = int(encoded["input_ids"].max().item())
        min_token_id = int(encoded["input_ids"].min().item())
        vocab_size = model.get_input_embeddings().num_embeddings
        if min_token_id < 0 or max_token_id >= vocab_size:
            raise ValueError(
                f"Token id range [{min_token_id}, {max_token_id}] is invalid for vocab size {vocab_size}."
            )

        seq_len = int(encoded["input_ids"].shape[1])
        max_positions = getattr(model.config, "max_position_embeddings", None)
        num_virtual_tokens = get_prompt_virtual_tokens(model)
        if max_positions is not None and (seq_len + 2 + num_virtual_tokens) > max_positions:
            raise ValueError(
                f"Sequence too long for model: seq_len={seq_len}, num_virtual_tokens={num_virtual_tokens}, "
                f"max_positions={max_positions}."
            )

    encoded = encoded.to(device)

    with torch.no_grad():
        logits = model(**encoded).logits
        preds = torch.argmax(logits, dim=-1)
    return preds.cpu().tolist()

def compute_binary_metric(metric, metric_name, predictions, references):
    common_kwargs = {
        "predictions": predictions,
        "references": references,
        "average": "binary",
        "pos_label": 1,
    }
    try:
        return metric.compute(**common_kwargs, zero_division=0)[metric_name]
    except TypeError:
        return metric.compute(**common_kwargs)[metric_name]

test_ds = dataset_dict["test"]
batch_size = 64
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
recall_metric = evaluate.load("recall")
precision_metric = evaluate.load("precision")
model_results = []

for spec in MODEL_SPECS:
    print(
        f"Evaluating {spec['display_name']} from {spec['repo']} ({spec['subfolder']})"
    )
    model, tokenizer = load_model_and_tokenizer(spec)
    max_input_len, max_positions, num_virtual_tokens = get_model_length_info(model)
    if max_positions is not None:
        print(
            f"  max_input_len={max_input_len} (max_positions={max_positions}, virtual_tokens={num_virtual_tokens})"
        )

    all_preds = []
    for i in range(0, len(test_ds), batch_size):
        batch_texts = test_ds[i:i + batch_size]["Text"]
        batch_preds = predict_batch(
            model,
            tokenizer,
            batch_texts,
            max_length=max_input_len,
        )
        all_preds.extend(batch_preds)

    references = test_ds["Label"]
    metric_values = {
        "accuracy": accuracy_metric.compute(
            predictions=all_preds,
            references=references,
        )["accuracy"],
        "f1": compute_binary_metric(
            f1_metric,
            "f1",
            all_preds,
            references,
        ),
        "recall": compute_binary_metric(
            recall_metric,
            "recall",
            all_preds,
            references,
        ),
        "precision": compute_binary_metric(
            precision_metric,
            "precision",
            all_preds,
            references,
        ),
    }

    model_results.append(
        {
            "display_name": spec["display_name"],
            "repo": spec["repo"],
            "subfolder": spec["subfolder"],
            **metric_values,
        }
    )
    print(
        f"  accuracy={metric_values['accuracy']:.4f} | "
        f"f1={metric_values['f1']:.4f} | "
        f"recall={metric_values['recall']:.4f} | "
        f"precision={metric_values['precision']:.4f}"
    )

    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nSummary (sorted by accuracy):")
for row in sorted(model_results, key=lambda x: x["accuracy"], reverse=True):
    print(
        f"- {row['display_name']} ({row['subfolder']}): "
        f"accuracy={row['accuracy']:.4f}, "
        f"f1={row['f1']:.4f}, "
        f"recall={row['recall']:.4f}, "
        f"precision={row['precision']:.4f}"
    )

Evaluating full_finetune_seed3 from BrianLeung/bertweet-sentiment140-full-finetune (seed3)


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


config.json:   0%|          | 0.00/936 [00:00<?, ?B/s]

seed3/model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  max_input_len=128 (max_positions=130, virtual_tokens=0)
  accuracy=0.7150 | f1=0.7107 | recall=0.7447 | precision=0.6796

Summary (sorted by accuracy):
- full_finetune_seed3 (seed3): accuracy=0.8950, f1=0.8955, recall=0.9574, precision=0.8411
- lora_r32_seed3 (r32/seed3): accuracy=0.8850, f1=0.8856, recall=0.9468, precision=0.8318
- lora_r8_seed3 (r8/seed3): accuracy=0.8850, f1=0.8856, recall=0.9468, precision=0.8318
- lora_r4_seed3 (r4/seed3): accuracy=0.8850, f1=0.8844, recall=0.9362, precision=0.8381
- prompt_tuning_seed3 (seed3): accuracy=0.8650, f1=0.8643, recall=0.9149, precision=0.8190
- classifier_only_seed3 (seed3): accuracy=0.7150, f1=0.7107, recall=0.7447, precision=0.6796


In [7]:
def classify_text_all_models(text):
    cleaned = replace(text)
    predictions = []

    for spec in MODEL_SPECS:
        model, tokenizer = load_model_and_tokenizer(spec)
        max_input_len = get_max_input_len(model)
        pred_id = predict_batch(
            model,
            tokenizer,
            [cleaned],
            max_length=max_input_len,
        )[0]
        label = id2label[pred_id]

        predictions.append(
            {
                "model": spec["display_name"],
                "repo": spec["repo"],
                "subfolder": spec["subfolder"],
                "predicted_label": label,
                "predicted_id": pred_id,
            }
        )

        del model
        del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return {
        "text": text,
        "cleaned_text": cleaned,
        "predictions": predictions,
    }

user_text = "I love how fast and reliable this model is!"
result = classify_text_all_models(user_text)

print(f"Text: {result['text']}")
print(f"Cleaned text: {result['cleaned_text']}")
print("\nPredictions:")
for item in result["predictions"]:
    print(
        f"- {item['model']} ({item['subfolder']}): {item['predicted_label']} ({item['predicted_id']})"
    )

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[trans

...

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[trans

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[trans

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Text: I love how fast and reliable this model is!
Cleaned text: I love how fast and reliable this model is!

Predictions:
- full_finetune_seed3 (seed3): positive (1)
- lora_r32_seed3 (r32/seed3): positive (1)
- lora_r8_seed3 (r8/seed3): positive (1)
- lora_r4_seed3 (r4/seed3): positive (1)
- prompt_tuning_seed3 (seed3): positive (1)
- classifier_only_seed3 (seed3): positive (1)
